# Day 37 — Dynamic Feature EDA

Exploratory analysis of the dynamic (behavioral) features extracted from
CAPEv2 reports, following the schema in `docs/dynamic_feature_spec.md` v1.0.

**Covers:**
- Sequence length distribution
- Most common API calls
- API category breakdown
- Class balance (benign vs malicious)
- Behavioral feature distributions

Since Day 35 (full corpus run) is pending, we demonstrate on the two
test fixtures: `cape_report_benign.json` and `cape_report_suspicious.json`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import json
from collections import Counter

from dynamic_sandbox.api_sequence import (
    extract_flat_sequence, api_category_counts, api_ngram_counts
)
from dynamic_sandbox.behavior_draft import extract_all_behavior_features
from dynamic_sandbox.build_dynamic_matrix import (
    extract_network_flags, extract_signature_features, extract_category_counts
)

In [ ]:
fixtures_dir = os.path.join('..', 'tests', 'fixtures')

with open(os.path.join(fixtures_dir, 'cape_report_benign.json')) as f:
    benign = json.load(f)
with open(os.path.join(fixtures_dir, 'cape_report_suspicious.json')) as f:
    suspicious = json.load(f)

reports = {
    'benign (notepad.exe)': benign,
    'suspicious (suspicious_sample.exe)': suspicious,
}
print(f'Loaded {len(reports)} fixture reports.')

## 1. Sequence Length Distribution

In [ ]:
print(f'{"Sample":<45} {"API Calls":>10} {"Processes":>10}')
print('-' * 67)
for label, report in reports.items():
    seq = extract_flat_sequence(report)
    n_procs = len(report.get('behavior', {}).get('processes', []))
    print(f'{label:<45} {len(seq):>10} {n_procs:>10}')

print()
print('Note: with a full corpus (Day 35), this section would show a')
print('histogram of sequence lengths and descriptive statistics.')
print('Typical malware: 50-500+ calls; benign apps: 5-50 calls.')

## 2. Most Common API Calls

In [ ]:
for label, report in reports.items():
    seq = extract_flat_sequence(report)
    counts = Counter(seq)
    print(f'\n--- {label} ---')
    print(f'Unique API calls: {len(counts)}')
    print(f'Top 10:')
    for api, count in counts.most_common(10):
        pct = 100 * count / len(seq)
        print(f'  {api:<35} {count:>4} ({pct:5.1f}%)')

## 3. API Category Breakdown

In [ ]:
for label, report in reports.items():
    cats = api_category_counts(report)
    total = sum(cats.values())
    print(f'\n--- {label} (total: {total} calls) ---')
    for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
        pct = 100 * count / total if total else 0
        bar = '#' * int(pct / 2)
        print(f'  {cat:<12} {count:>4} ({pct:5.1f}%) {bar}')

## 4. Behavioral Numeric Features Comparison

In [ ]:
print(f'{"Feature":<25} {"Benign":>10} {"Suspicious":>10} {"Delta":>10}')
print('-' * 57)

benign_feats = extract_all_behavior_features(benign)
susp_feats = extract_all_behavior_features(suspicious)

for key in benign_feats:
    b = benign_feats[key]
    s = susp_feats[key]
    delta = s - b
    marker = ' <<<' if abs(delta) > 0 else ''
    print(f'{key:<25} {b:>10} {s:>10} {delta:>+10}{marker}')

## 5. Network & Signature Indicators

In [ ]:
for label, report in reports.items():
    net = extract_network_flags(report)
    sigs = extract_signature_features(report)
    print(f'\n--- {label} ---')
    print(f'  Network: DNS={net["has_dns"]}  HTTP={net["has_http"]}  '
          f'TCP={net["has_tcp"]}  UDP={net["has_udp"]}')
    print(f'  Signatures: count={sigs["signature_count"]}, '
          f'max_severity={sigs["max_severity"]}')

## 6. Top Bigrams Comparison

In [ ]:
for label, report in reports.items():
    seq = extract_flat_sequence(report)
    bigrams = api_ngram_counts(seq, ns=(2,))
    print(f'\n--- {label} (top 10 bigrams) ---')
    for ng, count in bigrams.most_common(10):
        print(f'  {ng:<55} {count}')

## Key Observations

1. **Sequence length**: Suspicious sample has 2x+ the API calls of benign — 
   malware performs more system interactions (file drops, registry writes, process spawning).

2. **API diversity**: Suspicious sample uses more unique APIs, especially in the
   `process` and `network` categories that benign apps rarely touch.

3. **Category distribution**: Benign is dominated by `system` and `registry` reads;
   suspicious shows heavy `process`, `file`, `network`, and `registry` write activity.

4. **Process tree**: Benign has depth=1, breadth=0 (single process);
   suspicious has depth=3, breadth=2 (cmd→whoami, cmd→powershell chains).

5. **Network flags**: Benign has no network activity; suspicious has DNS + HTTP + TCP + UDP.

6. **Signatures**: Only suspicious triggers behavioral signatures (persistence, injection).

These separations suggest the 23-dim tabular vector alone has strong discriminative
power. The API sequence adds temporal ordering information that tabular features miss —
the LSTM/Transformer (Days 40-41) will capture patterns like
`NtAllocateVirtualMemory → NtWriteVirtualMemory → CreateRemoteThread` (injection chain).